# Notebook 10_b: Rule- and Taxonomy-Based Job Profile Expansion (Method 1.1)

This notebook is based on rule- and taxonomy-based approaches to skill extraction and job profile expansion from text data. The goal is to create a transparent, knowledge-based expansion of existing job profiles based on external text sources. Specific methodological references and literature are explained in the respective sections of the notebook (based on: Cenikj et al. 2021; Karakatsanis et al. 2017; Boselli et al. 2018; Zare et al. 2025; Senger et al. 2024).

In this notebook, base profiles for each KldB-5 position are expanded to include additional skills found in external data sources (e.g., job ads, profiles, scrapes, CSV files) but not yet included in the “Base Target” profile. 2 steps:
1. **Job Title Matching (external -> KldB):** Match job titles from an external document against KldB titles (DE/EN), always save the best score, and accept matches that meet a score cutoff. This allows the skills found in the document to be assigned to a KldB-5 code.
2. **Skill Extraction from External Documents:** Use the skill vocabulary (ESCO + BA + LinkedIn skills in a common ID notation from Notebook 10_a) and extract skills from the document text (rule- and dictionary-based using spaCy’s `EntityRuler`).

- External texts often contain specific requirements/technologies -> skill candidates are extracted from these
- These skills are mapped to KldB occupations
- Subsequently, the system checks which skills are new compared to the baseline SOLL (Novel Skills)
- The result is an expanded SOLL profile, including metadata for traceability

Outputs
- **Doc-Level**: KldB assignment per document + score + number of extracted skills  
- **KldB Skill Aggregation**: Which skills occur for each KldB and how frequently (`doc_freq`, `skill_count`)  
- **Novel Skills Table**: Skill candidates not included in the base SOLL  
- **Extended SOLL (Long & Agg)**: Base SOLL + additions, including marker columns and frequency metadata

Possible insights: Individual skills may not match perfectly; this can occur because:
  - Job title matching is heuristic (fuzzy matching, language, abbreviations)
  - External data often contains generic or employer-branding skills
  - LinkedIn skills are sometimes very broad
  
(Alternative notebooks and files located at: notebooks\10b-Alternative\10b-Cutoff_08+with_linkedin and 10b-Cutoff-09)

In [2]:
# Imports + Paths
from pathlib import Path
import re
import json
import pandas as pd
import numpy as np

import spacy
from spacy.pipeline import EntityRuler

from rapidfuzz import process, fuzz
from tqdm.notebook import tqdm
tqdm.pandas()

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_columns", 200)

# Project
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
DATA_INTERIM = DATA_DIR / "interim"
DATA_PROCESSED = DATA_DIR / "processed"
DATA_PROCESSED_EXTERNAL = DATA_DIR / "processed_external"

print("PROJECT_ROOT:", PROJECT_ROOT)

# Inputs
DOCS_PATH = DATA_PROCESSED_EXTERNAL / "documents_raw.parquet"
KDB_MAPPING_PATH = DATA_INTERIM / "kldb_esco_mapping.parquet"
KDB_SOLL_LONG_PATH = DATA_PROCESSED / "kldb_skills_soll_long.parquet"

# Skill-Vocab: Parquet from 10a
VOCAB_PARQUET_CANDIDATES = [
    DATA_PROCESSED_EXTERNAL / "skills_vocab_with_linkedin_with_skill_id_filtered.parquet", # preferred
    DATA_PROCESSED_EXTERNAL / "skills_vocab_with_linkedin_with_skill_id.parquet", # otherwise
]
CONCEPTS_PARQUET_CANDIDATES = [
    DATA_PROCESSED_EXTERNAL / "skills_concepts_filtered.parquet",
    DATA_PROCESSED_EXTERNAL / "skills_concepts.parquet",
]
VOCAB_CSV_FALLBACK = DATA_PROCESSED_EXTERNAL / "skills_vocab_with_linkedin.csv"

# Outputs (Test)
DOC_SKILLS_TEST_PATH = DATA_PROCESSED_EXTERNAL / "doc_skills_rulebased_test.parquet"
KDB_SKILLS_TEST_PATH = DATA_PROCESSED_EXTERNAL / "kldb_skills_rulebased_test.parquet"
KDB_EXTENSION_TEST_PATH = DATA_PROCESSED_EXTERNAL / "kldb_skill_rule_extension_test.parquet"

# Outputs (Full)
DOC_SKILLS_FULL_PATH = DATA_PROCESSED_EXTERNAL / "doc_skills_rulebased_full.parquet"
KDB_SKILLS_FULL_PATH = DATA_PROCESSED_EXTERNAL / "kldb_skills_rulebased_full.parquet"
KDB_EXTENSION_FULL_PATH = DATA_PROCESSED_EXTERNAL / "kldb_skill_rule_extension_full.parquet"

# Gold/Eval-Set
PRED_PATH = DATA_PROCESSED_EXTERNAL / "pred_skills_10b_rulebased.parquet"

PROJECT_ROOT: c:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung


## 1. Performance & Configuration Parameters

After expansion, the combined skills vocabulary (`skills_vocab_with_linkedin.csv`) contains:
- ESCO skills + alt labels
- BA competencies + synonyms
- LinkedIn skills (Employment Skills list)
--> a total of > 120,000 vocabulary entries.

At the same time, `documents_raw.parquet` contains approximately 290k documents (190k excluding Udemy): LinkedIn job postings, Workwise, CVs, profiles, and Udemy. Applying EntityRuler to all documents with all patterns may therefore result in long runtime.
To enable iterative processing, the following configuration parameters are defined here:
- `MAX_PATTERNS` specifies how many vocabulary patterns are loaded into the EntityRuler: `None` = all (maximum coverage, longest runtime).
- `MAX_DOCS_FOR_TEST` limits the number of documents processed: `None` = all documents.

For the first test iteration:
- `MAX_PATTERNS = 50,000`
- `MAX_DOCS_FOR_TEST = 2,000`

If performance is acceptable, these limits can be gradually increased or set to `None`. Various variants were tested here.

Performance Leverage (Test vs. Full Run)
- MAX_DOCS_FOR_TEST: limits the number of documents (e.g., 2,000 for fast test iterations).
- MAX_PATTERNS: Limits the number of skill patterns in EntityRuler (e.g., 50,000).
- USE_FILTERED_VOCAB: Uses the LinkedIn variant filtered in 10A to reduce generic skills.
- PIPELINE_ORDER: Match-->Extract

In [3]:
# Performance/Test Control
MAX_PATTERNS = None # None = all
MAX_DOCS_FOR_TEST = None # None = all
RANDOM_STATE = 42

# Skill-Extraktion: matches against LOWER, so patterns are case-insensitive
USE_FILTERED_VOCAB = True    # reduce LinkedIn-Noise
PHRASE_MATCHER_ATTR = "LOWER"

# Title Matching
MATCH_SCORE_MIN = 0.85  # initial value in the alternative Notebook 0.80; here, 0.85 is better (based on the results from Notebook 09 as well)
TITLE_SCORE_CUTOFF = int(MATCH_SCORE_MIN * 100)

# Check Pipeline Order
PIPELINE_ORDER = "match_then_extract" # test Match->Extract vs. Extract -> Match

print("MAX_PATTERNS:", MAX_PATTERNS)
print("MAX_DOCS_FOR_TEST:", MAX_DOCS_FOR_TEST)
print("USE_FILTERED_VOCAB:", USE_FILTERED_VOCAB)
print("TITLE_SCORE_CUTOFF:", TITLE_SCORE_CUTOFF, "(RapidFuzz 0-100)")

MAX_PATTERNS: None
MAX_DOCS_FOR_TEST: None
USE_FILTERED_VOCAB: True
TITLE_SCORE_CUTOFF: 85 (RapidFuzz 0-100)


`MATCH_SCORE_MIN = 0.85`: a deliberately strict cutoff for more robust title assignment (quality > coverage). Value selected based on the match quality analysis from Notebook 09 and Alternative 10b (notebooks\10-b-Alternative).

## 2. Load Skill Vocabulary & Quality Checks

Load the standardized skill vocabulary, which serves as a “dictionary” for skill extraction:
- **ESCO** (Skill URIs/Skill IDs)
- **BA** (e.g., BAK/BA labels)
- **LinkedIn Skills** (as an additional skill source)

The file is available as `skills_vocab_with_linkedin.csv` and contains:
- `vocab_id`: unique identifier for each vocabulary entry
- `label`: skill name/search term
- `source_system`: `“ESCO”`, `‘BA’`, or `“LINKEDIN”`
- `source_id`: ESCO skill URI or BA code; for LinkedIn, an artificial ID
- `lang`: `“en”` or `“de”`
- `label_type`: e.g., `preferred`, `alt`, `ba_label`, `ba_synonym`, `linkedin_skill`
- `label_norm`: normalized label
- `is_noise`: marked as noise in Notebook 10a
- All entries are merged using a common `skill_id` logic (e.g., prefixes `ESCO:`, `BA:`, `LINKEDIN:`) so that they can later be aggregated in a source-agnostic manner.
- For rule- and dictionary-based extraction, the skill labels are normalized (LOWER/`label_norm`) to ensure that matching is robust against case sensitivity and simple text variations.

In [4]:
# Load Skill vocabulary
def load_first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None

vocab_path = load_first_existing(VOCAB_PARQUET_CANDIDATES if USE_FILTERED_VOCAB else VOCAB_PARQUET_CANDIDATES[::-1])
concepts_path = load_first_existing(CONCEPTS_PARQUET_CANDIDATES if USE_FILTERED_VOCAB else CONCEPTS_PARQUET_CANDIDATES[::-1])

if vocab_path is not None:
    df_vocab = pd.read_parquet(vocab_path)
    print("Loaded vocab parquet:", vocab_path)
else:
    df_vocab = pd.read_csv(VOCAB_CSV_FALLBACK)
    print("Loaded vocab CSV fallback:", VOCAB_CSV_FALLBACK)

print("df_vocab shape:", df_vocab.shape)
display(df_vocab.head(3))
print(df_vocab.columns.tolist())

# Qualification Checks + Distribution
required_cols = {"skill_id", "label", "source_system", "lang", "label_type"}
missing = required_cols - set(df_vocab.columns)
assert len(missing) == 0, f"Vocab missing required columns: {missing}"

assert df_vocab["skill_id"].isna().sum() == 0, "Missing skill_id in vocab!"
assert df_vocab["label"].isna().sum() == 0, "Missing label in vocab!"

Loaded vocab parquet: c:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_external\skills_vocab_with_linkedin_with_skill_id_filtered.parquet
df_vocab shape: (126111, 9)


,vocab_id,label,source_system,source_id,lang,label_type,label_norm,is_noise,skill_id
0,ESCO:http://data.europa.eu/esco/skill/0005c151-5b5a-4a66-8aac-60e734beb1ab:preferred:manage musical staff,manage musical staff,ESCO,http://data.europa.eu/esco/skill/0005c151-5b5a-4a66-8aac-60e734beb1ab,en,preferred,manage musical staff,False,ESCO:http://data.europa.eu/esco/skill/0005c151-5b5a-4a66-8aac-60e734beb1ab
1,ESCO:http://data.europa.eu/esco/skill/00064735-8fad-454b-90c7-ed858cc993f2:preferred:supervise correctional procedures,supervise correctional procedures,ESCO,http://data.europa.eu/esco/skill/00064735-8fad-454b-90c7-ed858cc993f2,en,preferred,supervise correctional procedures,False,ESCO:http://data.europa.eu/esco/skill/00064735-8fad-454b-90c7-ed858cc993f2
2,ESCO:http://data.europa.eu/esco/skill/000709ed-2be5-4193-b056-45a97698d828:preferred:apply anti-oppressive practices,apply anti-oppressive practices,ESCO,http://data.europa.eu/esco/skill/000709ed-2be5-4193-b056-45a97698d828,en,preferred,apply anti-oppressive practices,False,ESCO:http://data.europa.eu/esco/skill/000709ed-2be5-4193-b056-45a97698d828


['vocab_id', 'label', 'source_system', 'source_id', 'lang', 'label_type', 'label_norm', 'is_noise', 'skill_id']


It is normal to have many labels per `skill_id` (synonyms, different sources/languages).

Creating display labels (one label per skill ID): A skill can have multiple labels (synonyms, languages, sources). However, for analyses and output tables, we need exactly one “display label” per `skill_id`. Therefore, the priority order is:
1. ESCO `preferred`
2. BA `ba_label`
3. Others (synonyms/additional sources)

Output `df_skill_display`: contains exactly one label per `skill_id` + metadata (source, language).

In [5]:
# Prioritize display labels by skill_id: ESCO preferred > BA label > others
def label_priority(row):
    ss = str(row["source_system"]).upper()
    lt = str(row["label_type"]).lower()
    # first ESCO preferred 
    if ss == "ESCO" and "preferred" in lt:
        return 0
    # then BA label
    if ss == "BA" and "ba_label" in lt:
        return 1
    # ESCO alt/BA syn/rest
    return 9

df_labels = df_vocab.copy()
df_labels["prio"] = df_labels.apply(label_priority, axis=1)

df_skill_display = (
    df_labels.sort_values(["skill_id", "prio"])
    .drop_duplicates(subset=["skill_id"])
    [["skill_id", "label", "source_system", "lang", "label_type"]]
    .rename(columns={
        "label": "display_label",
        "source_system": "display_source",
        "lang": "display_lang",
    })
)

print("df_skill_display shape:", df_skill_display.shape)
display(df_skill_display.head(10))

df_skill_display shape: (58303, 5)


,skill_id,display_label,display_source,display_lang,label_type
13939,BA:K 00,"Land-, Forstwirtschaft, Gartenbau",BA,de,ba_label
13941,BA:K 0001,Floristik,BA,de,ba_label
13942,BA:K 0001-000,Blumenversand,BA,de,ba_label
13945,BA:K 0001-001,Gestecke anfertigen,BA,de,ba_label
13952,BA:K 0001-002,Girlanden anfertigen (Floristik),BA,de,ba_label
13961,BA:K 0001-003,Hochzeitsfloristik,BA,de,ba_label
13967,BA:K 0001-004,Hydrokultur,BA,de,ba_label
13968,BA:K 0001-005,Ikebana,BA,de,ba_label
13973,BA:K 0001-006,"Kranzbinden, -flechten",BA,de,ba_label
13982,BA:K 0001-007,Pflanzen dekorativ arrangieren,BA,de,ba_label


Result = 58.303 unique skills in the skill vocabulary

## 3. Loading External Documents & Selecting Sources

Load external data (docs) using:
- `doc_id`
- `source_type`/`source_name`
- `job_title_raw`
- `raw_text`
- `language`

Then filter for the sources relevant to profile enrichment (e.g., job ads, scrapes, CSV, LinkedIn profiles). This is because some sources are unsuitable due to noise, duplicates, inappropriate text types, etc. The Udemy course dataset is not well-suited for profile enrichment anyway, due to missing job titles, and is therefore not used.

In [6]:
# Load only metadata (not raw_text due to RAM issues)
meta_cols = ["doc_id", "source_type", "source_name", "job_title_raw", "language"]
df_docs_meta = pd.read_parquet(DOCS_PATH, columns=meta_cols, engine="pyarrow")

print("df_docs_meta shape:", df_docs_meta.shape)
display(df_docs_meta.head(3))
display(df_docs_meta["source_type"].value_counts().head(20))

df_docs_meta shape: (292109, 5)


,doc_id,source_type,source_name,job_title_raw,language
0,921716,job_ad,kaggle_linkedin_2023_2024_big,Marketing Coordinator,en
1,1829192,job_ad,kaggle_linkedin_2023_2024_big,Mental Health Therapist/Counselor,en
2,10998357,job_ad,kaggle_linkedin_2023_2024_big,Assitant Restaurant Manager,en


source_type
job_ad              137161
course               98104
cv                   54933
linkedin_profile      1525
job_ad_scrape          386
Name: count, dtype: Int64

Filtering relevant sources: 

Exclusion of `course` (Udemy/Coursera): Course descriptions differ significantly from labor market/profile texts in terms of text type and purpose (marketing/learning objective phrasing instead of specific job requirements or job titles). They would systematically skew the skill distribution and are therefore not used for profile expansion in this method.

In [7]:
RELEVANT_SOURCE_TYPES = {"job_ad", "job_ad_scrape", "linkedin_profile", "cv",} # NOT “course” because it is not suitable for professional development, but with “cv” because the methodology is sound

df_docs_meta = df_docs_meta[df_docs_meta["source_type"].isin(RELEVANT_SOURCE_TYPES)].copy()

print("after source filter:", df_docs_meta.shape)
display(df_docs_meta["source_type"].value_counts())

after source filter: (194005, 5)


source_type
job_ad              137161
cv                   54933
linkedin_profile      1525
job_ad_scrape          386
Name: count, dtype: Int64

Test Run vs. Full Run: Test Run (fast): `df_docs_run` is a sample (2,000 documents); Full Run (slow): `df_docs_run` contains all relevant documents

In [8]:
if MAX_DOCS_FOR_TEST is not None:
    df_docs_run = df_docs_meta.sample(
        n=min(MAX_DOCS_FOR_TEST, len(df_docs_meta)),
        random_state=RANDOM_STATE
    ).copy()
else:
    df_docs_run = df_docs_meta.copy()

print("df_docs_meta_run:", df_docs_run.shape)
display(df_docs_run.head(3))

df_docs_meta_run: (194005, 5)


,doc_id,source_type,source_name,job_title_raw,language
0,921716,job_ad,kaggle_linkedin_2023_2024_big,Marketing Coordinator,en
1,1829192,job_ad,kaggle_linkedin_2023_2024_big,Mental Health Therapist/Counselor,en
2,10998357,job_ad,kaggle_linkedin_2023_2024_big,Assitant Restaurant Manager,en


## 4. Creating Matching Pools: KldB Job Titles (DE + EN Proxy)

External job titles should be matched to KldB job titles (not to KldB group designations such as kldb_title_de from the mapping). For this purpose, candidate pools:
- DE pool: original German job titles with kldb_5_code
- EN pool: EN proxy translations of these job titles, including kldb_5_code (newly generated file in Notebook 09)

Result: a standardized candidate data file with columns kldb_5_code, job_title, lang, and title_norm.

In [9]:
# Job titel Paths
# KldB Job Titles (DE)
KDB_JOB_TITLES_LONG_PATH = DATA_PROCESSED / "kldb_job_titles_long.parquet"
# EN Job Titles with KLDB 5 Code
KDB_TITLES_EN_PROXY_WITH_CODE_PATH = DATA_PROCESSED_EXTERNAL / "kldb_titles_en_proxy_with_code.parquet"

Load Job Titles (DE):

In [10]:
def pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

df_titles_de = pd.read_parquet(KDB_JOB_TITLES_LONG_PATH)
print("Loaded DE job titles:", df_titles_de.shape)
print(df_titles_de.columns.tolist())
display(df_titles_de.head(5))

col_code = pick_col(df_titles_de, ["kldb_5_code", "kldb5_code", "kldb_code"])
col_de   = pick_col(df_titles_de, ["job_title_de", "kldb_job_title_de", "job_title", "title_de"])

assert col_code is not None and col_de is not None, "DE job titles: required cols not found!"

df_titles_de = df_titles_de[[col_code, col_de]].rename(columns={col_code:"kldb_5_code", col_de:"job_title_de"})
df_titles_de["kldb_5_code"] = df_titles_de["kldb_5_code"].astype(str).str.extract(r"(\d{5})")[0]
df_titles_de["job_title_de"] = df_titles_de["job_title_de"].astype(str).str.strip()

df_titles_de = df_titles_de.dropna(subset=["kldb_5_code","job_title_de"])
df_titles_de = df_titles_de[df_titles_de["job_title_de"] != ""].drop_duplicates().reset_index(drop=True)

print("Clean DE titles:", df_titles_de.shape)
display(df_titles_de.head(10))

Loaded DE job titles: (18838, 2)
['job_title_de', 'kldb_5_code']


,job_title_de,kldb_5_code
0,3-D-Artist,23224
1,3-D-Designer/in,23223
2,3-D-Druck-Spezialist/in,27103
3,Abbrucharbeiter/in,32101
4,Abdichter/in (Dachdeckerei),33232


Clean DE titles: (18838, 2)


,kldb_5_code,job_title_de
0,23224,3-D-Artist
1,23223,3-D-Designer/in
2,27103,3-D-Druck-Spezialist/in
3,32101,Abbrucharbeiter/in
4,33232,Abdichter/in (Dachdeckerei)
5,33393,Abdichtungspolier/in (Bauwerks- und Asphaltabdichtung)
6,42324,Abfallbeauftragte/r
7,42313,Abfallberater/in
8,34301,Abfallbeseitiger/in
9,34333,Abfalltechniker/in


In [11]:
# Test
print("Unique KldB codes in DE pool:", df_titles_de["kldb_5_code"].nunique())
print("DE titles per code:")
display(df_titles_de.groupby("kldb_5_code")["job_title_de"].nunique().describe())

Unique KldB codes in DE pool: 1300
DE titles per code:


count    1300.000000
mean       14.490769
std        17.439147
min         1.000000
25%         4.000000
50%         9.000000
75%        17.250000
max       207.000000
Name: job_title_de, dtype: float64

Load EN-Proxy translations (EN KLDB occupational titles with codes):

In [12]:
df_proxy_code = pd.read_parquet(KDB_TITLES_EN_PROXY_WITH_CODE_PATH)
print("Loaded EN proxy-with-code:", df_proxy_code.shape)
print(df_proxy_code.columns.tolist())
display(df_proxy_code.head(5))

col_code2 = pick_col(df_proxy_code, ["kldb_5_code", "kldb5_code"])
col_proxy_en = pick_col(df_proxy_code, ["kldb_title_en_proxy", "job_title_en", "title_en_proxy", "title_en"])

assert col_code2 is not None and col_proxy_en is not None, "Proxy-with-code missing required cols!"

df_proxy_code = df_proxy_code[[col_code2, col_proxy_en]].rename(columns={col_code2:"kldb_5_code", col_proxy_en:"job_title_en"})
df_proxy_code["kldb_5_code"] = df_proxy_code["kldb_5_code"].astype(str).str.extract(r"(\d{5})")[0]
df_proxy_code["job_title_en"] = df_proxy_code["job_title_en"].astype(str).str.strip()

df_proxy_code = df_proxy_code.dropna(subset=["kldb_5_code","job_title_en"])
df_proxy_code = df_proxy_code[df_proxy_code["job_title_en"] != ""].drop_duplicates().reset_index(drop=True)

print("Clean EN proxy-with-code:", df_proxy_code.shape)
display(df_proxy_code.head(10))

Loaded EN proxy-with-code: (18838, 3)
['kldb_title_de', 'kldb_title_en_proxy', 'kldb_5_code']


,kldb_title_de,kldb_title_en_proxy,kldb_5_code
0,3-D-Artist,3D artist,23224
1,3-D-Designer/in,3D designer,23223
2,3-D-Druck-Spezialist/in,3D printing specialist,27103
3,Abbrucharbeiter/in,Demolition worker,32101
4,Abdichter/in (Dachdeckerei),Sealer (roofing),33232


Clean EN proxy-with-code: (18090, 2)


,kldb_5_code,job_title_en
0,23224,3D artist
1,23223,3D designer
2,27103,3D printing specialist
3,32101,Demolition worker
4,33232,Sealer (roofing)
5,33393,Sealing polisher (building and asphalt sealing)
6,42324,Waste representative
7,42313,Waste consultant
8,34301,Waste disposer
9,34333,Waste technician


In [13]:
# Quality Check
missing = df_proxy_code["kldb_5_code"].isna().sum()
print("Missing kldb_5_code in proxy-with-code:", missing)

multi = df_proxy_code.groupby("job_title_en")["kldb_5_code"].nunique().sort_values(ascending=False).head(10)
print("Top 10 EN titles with multiple codes:")
display(multi)

Missing kldb_5_code in proxy-with-code: 0
Top 10 EN titles with multiple codes:


job_title_en
None                           30
Hydraulic engineer              5
Electrician                     4
Hotel manager                   3
Blacksmith                      3
Lifeguard                       3
Health insurance specialist     3
Advertising manager             3
Advertising designer            3
Interior designer               3
Name: kldb_5_code, dtype: int64

Building a Candidate Pool (EN + DE):

In [14]:
df_candidates_de = df_titles_de.rename(columns={"job_title_de":"job_title"}).copy()
df_candidates_de["lang"] = "de"

df_candidates_en = df_proxy_code.rename(columns={"job_title_en":"job_title"})[["kldb_5_code","job_title"]].copy()
df_candidates_en["lang"] = "en"

df_candidates = pd.concat([df_candidates_de, df_candidates_en], ignore_index=True)
df_candidates["job_title"] = df_candidates["job_title"].astype(str).str.strip()
df_candidates = df_candidates.dropna(subset=["kldb_5_code","job_title"])
df_candidates = df_candidates[df_candidates["job_title"] != ""].drop_duplicates().reset_index(drop=True)

print("Candidates total:", df_candidates.shape)
display(df_candidates["lang"].value_counts())
display(df_candidates.head(10))

Candidates total: (36928, 3)


lang
de    18838
en    18090
Name: count, dtype: int64

,kldb_5_code,job_title,lang
0,23224,3-D-Artist,de
1,23223,3-D-Designer/in,de
2,27103,3-D-Druck-Spezialist/in,de
3,32101,Abbrucharbeiter/in,de
4,33232,Abdichter/in (Dachdeckerei),de
5,33393,Abdichtungspolier/in (Bauwerks- und Asphaltabdichtung),de
6,42324,Abfallbeauftragte/r,de
7,42313,Abfallberater/in,de
8,34301,Abfallbeseitiger/in,de
9,34333,Abfalltechniker/in,de


In [15]:
# Output-Path
CANDIDATES_PATH = (DATA_INTERIM / "kldb_job_title_candidates_de_en.parquet")
CANDIDATES_PATH.parent.mkdir(parents=True, exist_ok=True)

# Save
df_candidates.to_parquet(CANDIDATES_PATH, index=False)
print("Saved candidates pool:")
print(" ", CANDIDATES_PATH)

Saved candidates pool:
  c:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\interim\kldb_job_title_candidates_de_en.parquet


In [16]:
df_candidates.to_csv( # as csv
    CANDIDATES_PATH.with_suffix(".csv"),
    index=False)

## 5. Title Cleaning & Normalization (for Better Matching)

Job titles from external sources are often noisy:
- Additions such as `(m/f/d)`, `Remote`, `Hybrid`, `Full-time`, location details
- Special characters, hyphens, abbreviations
- Mixed languages

`norm_title()` reduces noise to make fuzzy matching more stable. KLDB titles are standardized, so no major adjustments are needed, unlike with external datasets. Cleaning has a major impact on quality. This is a good place to make improvements (without changing the methodology) for adjustments, better quality, etc. Match against `title_norm`.
KLDB job titles are already clean; here, they are only slightly normalized to make matching more robust without losing meaning. External job titles are significantly noisier. Here, additional artifacts (which are still present/have not been removed) are used to improve match quality.

In [17]:
import re

# KldB Title
def norm_candidate(title: str) -> str:
    if not isinstance(title, str):
        return ""
    s = title.strip().lower()
    # Remove content within parentheses
    s = re.sub(r"\(.*?\)|\[.*?\]|\{.*?\}", " ", s)
    # Standardize delimiters
    s = re.sub(r"[/|•·–—_,:;]+", " ", s)
    # Keep only the relevant characters
    s = re.sub(r"[^a-z0-9äöüß\s\-]", " ", s)
    # Multiple spaces
    s = re.sub(r"\s+", " ", s).strip()
    return s

# External job titles: normalize them more
STOP_TOKENS = {
    # gender
    "mwd","m/w/d","w/m/d","d/m/w","m-f-d","f/m/d",
    # work mode / location noise
    "remote","hybrid","home","office","onsite","on-site",
    # employment type
    "fulltime","full-time","parttime","part-time","intern","internship","werkstudent","working","student",
    # seniority
    "junior","senior","lead","principal","head","manager","specialist","expert",
    # common extras
    "gn","gn*", "mw", "m", "w", "d",
}

def norm_query(title: str) -> str:
    if not isinstance(title, str):
        return ""
    s = title.strip().lower()
    # Remove the brackets
    s = re.sub(r"\(.*?\)|\[.*?\]|\{.*?\}", " ", s)
    # Delimiter
    s = re.sub(r"[/|•·–—_,:;]+", " ", s)
    # Remove special characters
    s = re.sub(r"[^a-z0-9äöüß\s\-]", " ", s)
    # Multiple spaces
    s = re.sub(r"\s+", " ", s).strip()
    # Tokenfilter
    tokens = []
    for t in s.split():
        t_clean = t.strip("-")
        if not t_clean:
            continue
        if t_clean in STOP_TOKENS:
            continue
        tokens.append(t_clean)

    return " ".join(tokens).strip()

Normalizing KLDB Candidates (DE + EN):

In [18]:
# Normalize candidates
df_candidates["title_norm"] = df_candidates["job_title"].apply(norm_candidate)

# empty it out
df_candidates = df_candidates[df_candidates["title_norm"] != ""].drop_duplicates(
    subset=["kldb_5_code", "lang", "title_norm"]
).reset_index(drop=True)

print("Candidates after light norm:", df_candidates.shape)
display(df_candidates.head(10))
display(df_candidates["lang"].value_counts())

Candidates after light norm: (35366, 4)


,kldb_5_code,job_title,lang,title_norm
0,23224,3-D-Artist,de,3-d-artist
1,23223,3-D-Designer/in,de,3-d-designer in
2,27103,3-D-Druck-Spezialist/in,de,3-d-druck-spezialist in
3,32101,Abbrucharbeiter/in,de,abbrucharbeiter in
4,33232,Abdichter/in (Dachdeckerei),de,abdichter in
5,33393,Abdichtungspolier/in (Bauwerks- und Asphaltabdichtung),de,abdichtungspolier in
6,42324,Abfallbeauftragte/r,de,abfallbeauftragte r
7,42313,Abfallberater/in,de,abfallberater in
8,34301,Abfallbeseitiger/in,de,abfallbeseitiger in
9,34333,Abfalltechniker/in,de,abfalltechniker in


lang
de    18103
en    17263
Name: count, dtype: int64

In [19]:
# Check: Different standard titles for each code
tmp_stats = df_candidates.groupby(["lang"])["kldb_5_code"].nunique()
print("Unique KldB codes in pools:", tmp_stats.to_dict())

Unique KldB codes in pools: {'de': 1300, 'en': 1300}


Normalize external job titles; apply stronger normalization here:

Later, match against `job_title_raw` from `df_docs_run`. Two columns: `job_title_norm_query` and language from `lang`.

In [20]:
df_docs_run["job_title_norm_query"] = df_docs_run["job_title_raw"].apply(norm_query)

# language is reliable -> language_guess = language
df_docs_run["language_guess"] = df_docs_run["language"]

print("Docs with non-empty query norm:", (df_docs_run["job_title_norm_query"]!="").mean())
display(df_docs_run[["job_title_raw","job_title_norm_query","language","language_guess"]].head(10))

Docs with non-empty query norm: 0.9985361202030876


,job_title_raw,job_title_norm_query,language,language_guess
0,Marketing Coordinator,marketing coordinator,en,en
1,Mental Health Therapist/Counselor,mental health therapist counselor,en,en
2,Assitant Restaurant Manager,assitant restaurant,en,en
3,Senior Elder Law / Trusts and Estates Associate Attorney,elder law trusts and estates associate attorney,en,en
4,Service Technician,service technician,en,en
5,Economic Development and Planning Intern,economic development and planning,en,en
6,Producer,producer,en,en
7,Building Engineer,building engineer,en,en
8,Respiratory Therapist,respiratory therapist,en,en
9,Worship Leader,worship leader,en,en


## 6. Job Titles & KldB Matching (with RapidFuzz)

Candidate pools, two pools:
- DE pool: KldB job titles (German)
- EN pool: English proxy titles that are already assigned a `kldb_5_code`, since many EN data sets are external

Matching strategy: First, exact match on normalized titles, followed by fuzzy matching (RapidFuzz `WRatio`) against the appropriate language pool (DE/EN), accepted starting at the score cutoff.
1. Exact match on normalized titles (fast & simple)
2. If no match: Fuzzy matching with RapidFuzz (WRatio) against the appropriate language pool
3. If language is unclear: select both pools and choose the best result

This approach follows text-mining methods for the labor market, in which job postings are assigned to standardized occupational classifications to bridge the gap between market reality and taxonomies (e.g., O*NET/ISCO) before structured requirements/skills are derived (Karakatsanis et al., 2017; Boselli et al., 2018).

Interpretation
- 100: exact match (very reliable)
- ≥ cutoff (e.g., 80/85): accepted fuzzy match; the higher the score, the better the fit, but fewer matches 
- Just below the cutoff often indicates a similar occupation or a more generic title  
- Very generic titles (e.g., Specialist, Manager, Professional) can generate high scores

This matching is heuristic but practical as a baseline. The assignment of documents to occupational classifications represents a simplified form of a knowledge-based matching problem, as described, for example, in the context of e-recruitment systems (Freire & de Castro, 2021). Such pure exact-match approaches reach their limits when dealing with different spelling variations (e.g., Boselli et al., 2018; Karakatsanis et al., 2017; Senger et al., 2024). Here, Method 1.1 deliberately remains a lexical, dictionary-based baseline approach (Cenikj et al., 2021).

In [21]:
# Pools + Exact-Index
TITLE_SCORE_CUTOFF = int(MATCH_SCORE_MIN * 100)  # first 80, then 85

cand_de = df_candidates[df_candidates["lang"]=="de"][["kldb_5_code","job_title","title_norm"]].reset_index(drop=True)
cand_en = df_candidates[df_candidates["lang"]=="en"][["kldb_5_code","job_title","title_norm"]].reset_index(drop=True)

de_norm_list = cand_de["title_norm"].tolist()
en_norm_list = cand_en["title_norm"].tolist()

# title_norm -> (code, original_title), if there are multiple, take the first one; otherwise, save the list
exact_de = cand_de.drop_duplicates("title_norm").set_index("title_norm")[["kldb_5_code","job_title"]].to_dict("index")
exact_en = cand_en.drop_duplicates("title_norm").set_index("title_norm")[["kldb_5_code","job_title"]].to_dict("index")

print("Pools:", len(cand_de), "DE |", len(cand_en), "EN")

Pools: 18103 DE | 17263 EN


Match-Funktion (Exact -> Fuzzy):

In [22]:
def _fuzzy_from_pool(q_norm, pool_norm_list, pool_df, pool_lang):
    if not q_norm or not pool_norm_list:
        return (None, None, np.nan, False, pool_lang, "none")

    best = process.extractOne(q_norm, pool_norm_list, scorer=fuzz.WRatio)
    if best is None:
        return (None, None, np.nan, False, pool_lang, "none")

    _, score, idx = best[0], float(best[1]), int(best[2])
    code = pool_df.loc[idx, "kldb_5_code"]
    title = pool_df.loc[idx, "job_title"]
    ok = score >= TITLE_SCORE_CUTOFF
    return (code, title, score, ok, pool_lang, "fuzzy")


def match_kldb(job_title_raw: str, lang: str = None):
    q_norm = norm_query(job_title_raw)
    if not q_norm:
        return (None, None, np.nan, False, None, "none")

    lang_eff = lang if lang in ("de","en") else None

    # Exact to normalized
    if lang_eff == "de":
        hit = exact_de.get(q_norm)
        if hit:
            return (hit["kldb_5_code"], hit["job_title"], 100.0, True, "de", "exact")
        return _fuzzy_from_pool(q_norm, de_norm_list, cand_de, "de")

    if lang_eff == "en":
        hit = exact_en.get(q_norm)
        if hit:
            return (hit["kldb_5_code"], hit["job_title"], 100.0, True, "en", "exact")
        return _fuzzy_from_pool(q_norm, en_norm_list, cand_en, "en")

    # unknown: Try both, prefer exact
    hit_de = exact_de.get(q_norm)
    hit_en = exact_en.get(q_norm)
    if hit_de and not hit_en:
        return (hit_de["kldb_5_code"], hit_de["job_title"], 100.0, True, "de", "exact")
    if hit_en and not hit_de:
        return (hit_en["kldb_5_code"], hit_en["job_title"], 100.0, True, "en", "exact")
    if hit_de and hit_en:
        # the same standard in both pools
        return (hit_de["kldb_5_code"], hit_de["job_title"], 100.0, True, "de", "exact")

    res_de = _fuzzy_from_pool(q_norm, de_norm_list, cand_de, "de")
    res_en = _fuzzy_from_pool(q_norm, en_norm_list, cand_en, "en")

    # The highest score wins
    s_de, s_en = res_de[2], res_en[2]
    if np.isnan(s_de) and np.isnan(s_en):
        return (None, None, np.nan, False, None, "none")
    if np.isnan(s_en) or (not np.isnan(s_de) and s_de >= s_en):
        return res_de
    return res_en

Apply matching to df_docs_run:

In [23]:
from tqdm.notebook import tqdm
tqdm.pandas()

tmp = df_docs_run[["doc_id","job_title_raw","language_guess"]].copy()

tmp[["kldb_5_code","kldb_match_title","kldb_match_score","has_kldb_match","match_lang","match_method"]] = tmp.progress_apply(
    lambda r: pd.Series(match_kldb(r["job_title_raw"], r["language_guess"])),
    axis=1
)

print("Score describe:")
display(tmp["kldb_match_score"].describe())
print("Accepted:", tmp["has_kldb_match"].sum(), "of", len(tmp))

df_docs_run = df_docs_run.merge(
    tmp[["doc_id","kldb_5_code","kldb_match_title","kldb_match_score","has_kldb_match","match_lang","match_method"]],
    on="doc_id", how="left"
)

display(df_docs_run[["doc_id","job_title_raw","language_guess","kldb_5_code","has_kldb_match","kldb_match_score","match_method"]].head(15))

  0%|          | 0/194005 [00:00<?, ?it/s]

Score describe:


count    193721.000000
mean         89.501213
std           5.496303
min          25.714286
25%          85.500000
50%          90.000000
75%          90.000000
max         100.000000
Name: kldb_match_score, dtype: float64

Accepted: 187954 of 194005


,doc_id,job_title_raw,language_guess,kldb_5_code,has_kldb_match,kldb_match_score,match_method
0,921716,Marketing Coordinator,en,92113,True,100.000000,exact
1,1829192,Mental Health Therapist/Counselor,en,84184,True,90.000000,fuzzy
2,10998357,Assitant Restaurant Manager,en,63301,True,92.564103,fuzzy
3,23221523,Senior Elder Law / Trusts and Estates Associate Attorney,en,92113,True,85.500000,fuzzy
4,35982263,Service Technician,en,25132,True,100.000000,exact
5,91700727,Economic Development and Planning Intern,en,12143,True,85.500000,fuzzy
6,103254301,Producer,en,94404,True,100.000000,exact
7,112576855,Building Engineer,en,25244,True,89.473684,fuzzy
8,1218575,Respiratory Therapist,en,81733,True,100.000000,exact
9,2264355,Worship Leader,en,24422,True,90.000000,fuzzy


- An average score of 89 is very good, 
- even with a cutoff of 85, there are many accepted matches - a good sign of coverage - despite the higher score (the same applies to cutoffs of 0.80 and 0.90)
- Most matches are very good; some are plausible or average, but there are also a few incorrect assignments (e.g., Senior Elder Law → Marketing or Inside Customer Service Associate → Restaurant Manager)

In [24]:
# Matching Analysis
n_all = len(df_docs_run)
n_ok = int(df_docs_run["has_kldb_match"].sum())
print(f"Match-Rate: {n_ok}/{n_all} = {n_ok/n_all:.2%}")

print("\nMatch-Methoden:")
display(df_docs_run["match_method"].value_counts(dropna=False).to_frame("count"))

print("\nMatch-Rate nach Sprache:")
display(
    df_docs_run.groupby("language_guess")["has_kldb_match"]
    .mean()
    .sort_values(ascending=False)
    .to_frame("match_rate")
)

if "source_type" in df_docs_run.columns:
    print("\nMatch-Rate nach Quelle:")
    display(
        df_docs_run.groupby("source_type")["has_kldb_match"]
        .mean()
        .sort_values(ascending=False)
        .to_frame("match_rate"))

Match-Rate: 187954/194005 = 96.88%

Match-Methoden:


,count
match_method,
fuzzy,174089
exact,19632
none,284



Match-Rate nach Sprache:


,match_rate
language_guess,
da,1.000000
et,1.000000
sk,1.000000
uk,1.000000
vi,1.000000
en,0.971287
cs,0.947368
es,0.930657
de,0.917950



Match-Rate nach Quelle:


,match_rate
source_type,
cv,0.984381
job_ad,0.962854
linkedin_profile,0.960000
job_ad_scrape,0.904145


(Match rate is very high at 0.80 and 0.85, but only 60% at 0.90)

## 7. Preparing Skill Extraction (Rule-Based) with spaCy EntityRuler

Setting up a minimalist spaCy pipeline based on the skill vocabulary (df_vocab):
- Tokenization + `EntityRuler`
- Patterns are based on skill labels from the vocabulary
- Matching is set to `LOWER` (case-insensitive) to make spelling variations in the text less critical
Output: Number of prepared patterns, active pipes in spaCy. Skill extraction is rule-based using a predefined skill lexicon and thus corresponds to a dictionary-based NER approach, as proposed, among other things, for building organizational skill inventories (Cenikj et al., 2021; Boselli et al., 2018; Zare et al., 2025).

In [25]:
# Minimal spaCy pipeline object (tokenization + EntityRuler)

# neutral (mixed German/English)
nlp = spacy.blank("xx")

ruler = nlp.add_pipe("entity_ruler",config={"phrase_matcher_attr": PHRASE_MATCHER_ATTR, "overwrite_ents": True})

# Build Patterns
df_patterns_src = df_vocab[["skill_id", "label"]].drop_duplicates().copy()
df_patterns_src["label_len"] = df_patterns_src["label"].astype(str).str.len()
df_patterns_src = df_patterns_src.sort_values("label_len", ascending=False)

if MAX_PATTERNS is not None:
    df_patterns_src = df_patterns_src.head(MAX_PATTERNS).copy()

patterns = [{"label": "SKILL", "pattern": r.label, "id": r.skill_id} for r in df_patterns_src.itertuples(index=False)]
ruler.add_patterns(patterns)

print("Patterns:", len(patterns))
print("Pipeline:", nlp.pipe_names)

# test
test_text = "Looking for Surface Water Hydrology, analyse transport business networks, Stock market Analysis and Jugendpsychiatrie (Pflege) experience." # Begriffe gewählt die auch im Test-Lauf vorkommen
doc = nlp(test_text)
hits = sorted(set(ent.ent_id_ for ent in doc.ents if ent.label_=="SKILL" and ent.ent_id_))
print("Smoke hits (first 20):", hits[:20], "| n=", len(hits))

Patterns: 126111
Pipeline: ['entity_ruler']
Smoke hits (first 20): ['BA:K 090203-031', 'ESCO:http://data.europa.eu/esco/skill/b8316819-feca-403d-902f-a86562fd91e9', 'LINKEDIN:stock market analysis', 'LINKEDIN:surface water hydrology'] | n= 4


Extraction smoke test: Short sample text test. Expectation: If `hits` is empty, that may still be okay (if no skill labels match exactly), especially during the test

**Document-level extraction:**

extract_skills_rulebased(text) returns a unique list of skill_ids for each document. The actual processing takes place in batches in the next step, in a RAM-safe manner.

In [26]:
MAX_CHARS = 50_000  # limit

def extract_skills_rulebased(text: str):
    if not isinstance(text, str) or not text.strip():
        return []
    if len(text) > MAX_CHARS:
        text = text[:MAX_CHARS]
    doc = nlp(text)
    ids = [ent.ent_id_ for ent in doc.ents if ent.label_ == "SKILL" and ent.ent_id_]
    return sorted(set(ids))

print(extract_skills_rulebased("Erfahrung mit Jugendpsychiatrie (Pflege), Python und Projektmanagement ist hilfreich.")[:30])

['BA:K 030300-058', 'BA:K 0705-218', 'BA:K 090203-031']


## 8. Batch Skill Extraction Only for Matched Docs (RAM-Safe, Drop-in Replacement)

Skill extraction is not run through a large dataset filter (which caused RAM spikes with large Parquet files), but is instead read in batches directly via Parquet RowGroups and filtered by `match_set` only within the batch.

Advantages:
- More stable against `ArrowMemoryError` (RowGroup-based streaming)
- Controllable batch size (`BATCH_ROWS`, fallback `BATCH_ROWS_FALLBACK`)
- Individual RowGroups can be skipped (with logging) instead of the entire run aborting

In [27]:
# Setup
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.compute as pc

# Parameter
MAX_CHARS = 30_000 # 30k should be more than enough; this will shorten very long raw_text strings. What about 20k for full? Otherwise, there will be RAM issues.
BATCH_ROWS = 256  # faster; otherwise -> 128
BATCH_ROWS_FALLBACK = 16 # for ArrowMemoryError within RowGroup

DOCS_PATH = str(DOCS_PATH)  # documents_raw.parquet path

# IDs: Extract only matched documents
match_ids = df_docs_run.loc[df_docs_run["has_kldb_match"]==True, "doc_id"].astype(str).tolist()
match_set = set(match_ids)

print("Docs to extract:", len(match_ids))

Docs to extract: 187954


Batch extraction from Parquet (without `dataset.filter`):
- Read `raw_text` in small batches directly from `ParquetFile`
- No `dataset.filter` to reduce Arrow RAM spikes
- Use smaller batches if necessary

In [28]:
def truncate_arrow_utf8(col_text, max_chars=MAX_CHARS): # Convert in Arrow to Python strings (RAM-safe)
    try:
        return pc.utf8_slice_codeunits(col_text, start=0, stop=max_chars)
    except Exception:
        return col_text

pf = pq.ParquetFile(DOCS_PATH)
print("Row groups:", pf.num_row_groups)

skill_rows = [] # (doc_id, skills_rule_ids, n_skills)
bad_docs = [] # Skip the documentation if an error occurs

def process_rowgroup(rg: int, batch_rows: int): # A row group with a fixed batch size
    for rb in pf.iter_batches(
        row_groups=[rg],
        columns=["doc_id", "raw_text"],
        batch_size=batch_rows,
        use_threads=False
    ):
        col_id = rb.column(0)
        col_tx = rb.column(1)

        # truncate before Python conversion
        col_tx_short = truncate_arrow_utf8(col_tx, MAX_CHARS)

        for i in range(rb.num_rows):
            did = col_id[i].as_py()
            if did is None:
                continue
            did = str(did)
            if did not in match_set:
                continue

            try:
                text = col_tx_short[i].as_py() or ""
            except pa.ArrowMemoryError:
                bad_docs.append(did)
                continue

            skills = extract_skills_rulebased(text)
            skill_rows.append((did, skills, len(skills)))

# loop
for rg in tqdm(range(pf.num_row_groups), desc="RowGroups"):
    try:
        process_rowgroup(rg, BATCH_ROWS)
    except pa.ArrowMemoryError as e:
        print(f"[RG {rg}] ArrowMemoryError -> fallback smaller batches. {e}")
        try:
            process_rowgroup(rg, BATCH_ROWS_FALLBACK)
        except pa.ArrowMemoryError as e2:
            print(f"[RG {rg}] Still failing -> skip rowgroup. {e2}")

print("Done. Extracted rows:", len(skill_rows))
print("Bad docs skipped:", len(set(bad_docs)))
if bad_docs:
    print("Example bad doc_ids:", list(dict.fromkeys(bad_docs))[:10])

Row groups: 1


RowGroups:   0%|          | 0/1 [00:00<?, ?it/s]

Done. Extracted rows: 187954
Bad docs skipped: 0


Output:
- `RowGroups: X` shows how many RowGroups are present in the Parquet file (Full Run > 1 (Test = 1))
- `Extracted rows: ...` corresponds to the approximate number of matched documents (- problematic documents)
- `Bad docs skipped: 0` (In a Full Run, a small number of problematic documents is tolerable)

Merge back into df_docs_run + checks:

In [29]:
df_sk = pd.DataFrame(skill_rows, columns=["doc_id","skills_rule_ids","n_skills"])

# Make the merge safe
df_docs_run["doc_id"] = df_docs_run["doc_id"].astype(str)
df_docs_run = df_docs_run.merge(df_sk, on="doc_id", how="left")

# defaults
df_docs_run["skills_rule_ids"] = df_docs_run["skills_rule_ids"].apply(lambda x: x if isinstance(x, list) else [])
df_docs_run["n_skills"] = df_docs_run["n_skills"].fillna(0).astype(int)

print("Docs with >=1 skill:", (df_docs_run["n_skills"] > 0).sum())
display(df_docs_run[["doc_id","job_title_raw","kldb_5_code","kldb_match_score","n_skills"]].head(10))

Docs with >=1 skill: 187774


,doc_id,job_title_raw,kldb_5_code,kldb_match_score,n_skills
0,921716,Marketing Coordinator,92113,100.000000,47
1,1829192,Mental Health Therapist/Counselor,84184,90.000000,40
2,10998357,Assitant Restaurant Manager,63301,92.564103,16
3,23221523,Senior Elder Law / Trusts and Estates Associate Attorney,92113,85.500000,25
4,35982263,Service Technician,25132,100.000000,6
5,91700727,Economic Development and Planning Intern,12143,85.500000,53
6,103254301,Producer,94404,100.000000,15
7,112576855,Building Engineer,25244,89.473684,69
8,1218575,Respiratory Therapist,81733,100.000000,47
9,2264355,Worship Leader,24422,90.000000,44


Example output of a title<->KldDB match. 187954 records had a KldB match and were processed.

## 9. Debug Overview: Pipeline Counts & Quick Checks

- Number of documents in the run
- Where data goes:
  - Total documents
  - Number of matches
  - Number of documents with ≥1 skill
  - Number of skill lines in long format
- `docs_with_skills ≤ docs_matched` -> Skills are extracted only for matched documents

In [30]:
# Pipeline-Counts
n_docs_total = len(df_docs_run)
n_matched = int(df_docs_run["has_kldb_match"].fillna(False).sum())
n_with_skills = int((df_docs_run["n_skills"].fillna(0) > 0).sum())

print("Docs total:", n_docs_total)
print("Docs matched (has_kldb_match):", n_matched)
print("Docs with >=1 skill:", n_with_skills)

# sanity: n_with_skills must not be greater than n_matched
assert n_with_skills <= n_matched, "Unexpected: more docs with skills than matched docs"

Docs total: 194005
Docs matched (has_kldb_match): 187954
Docs with >=1 skill: 187774


In [31]:
# Examples of Matched Docs
cols_show = ["doc_id","source_type","source_name","job_title_raw","language_guess","kldb_5_code","kldb_match_score","match_lang","match_method","n_skills"]

display(df_docs_run.loc[df_docs_run["has_kldb_match"]==True, cols_show].head(15))
display(df_docs_run.loc[df_docs_run["n_skills"]>0, cols_show].head(15))

,doc_id,source_type,source_name,job_title_raw,language_guess,kldb_5_code,kldb_match_score,match_lang,match_method,n_skills
0,921716,job_ad,kaggle_linkedin_2023_2024_big,Marketing Coordinator,en,92113,100.000000,en,exact,47
1,1829192,job_ad,kaggle_linkedin_2023_2024_big,Mental Health Therapist/Counselor,en,84184,90.000000,en,fuzzy,40
2,10998357,job_ad,kaggle_linkedin_2023_2024_big,Assitant Restaurant Manager,en,63301,92.564103,en,fuzzy,16
3,23221523,job_ad,kaggle_linkedin_2023_2024_big,Senior Elder Law / Trusts and Estates Associate Attorney,en,92113,85.500000,en,fuzzy,25
4,35982263,job_ad,kaggle_linkedin_2023_2024_big,Service Technician,en,25132,100.000000,en,exact,6
5,91700727,job_ad,kaggle_linkedin_2023_2024_big,Economic Development and Planning Intern,en,12143,85.500000,en,fuzzy,53
6,103254301,job_ad,kaggle_linkedin_2023_2024_big,Producer,en,94404,100.000000,en,exact,15
7,112576855,job_ad,kaggle_linkedin_2023_2024_big,Building Engineer,en,25244,89.473684,en,fuzzy,69
8,1218575,job_ad,kaggle_linkedin_2023_2024_big,Respiratory Therapist,en,81733,100.000000,en,exact,47
9,2264355,job_ad,kaggle_linkedin_2023_2024_big,Worship Leader,en,24422,90.000000,en,fuzzy,44


,doc_id,source_type,source_name,job_title_raw,language_guess,kldb_5_code,kldb_match_score,match_lang,match_method,n_skills
0,921716,job_ad,kaggle_linkedin_2023_2024_big,Marketing Coordinator,en,92113,100.000000,en,exact,47
1,1829192,job_ad,kaggle_linkedin_2023_2024_big,Mental Health Therapist/Counselor,en,84184,90.000000,en,fuzzy,40
2,10998357,job_ad,kaggle_linkedin_2023_2024_big,Assitant Restaurant Manager,en,63301,92.564103,en,fuzzy,16
3,23221523,job_ad,kaggle_linkedin_2023_2024_big,Senior Elder Law / Trusts and Estates Associate Attorney,en,92113,85.500000,en,fuzzy,25
4,35982263,job_ad,kaggle_linkedin_2023_2024_big,Service Technician,en,25132,100.000000,en,exact,6
5,91700727,job_ad,kaggle_linkedin_2023_2024_big,Economic Development and Planning Intern,en,12143,85.500000,en,fuzzy,53
6,103254301,job_ad,kaggle_linkedin_2023_2024_big,Producer,en,94404,100.000000,en,exact,15
7,112576855,job_ad,kaggle_linkedin_2023_2024_big,Building Engineer,en,25244,89.473684,en,fuzzy,69
8,1218575,job_ad,kaggle_linkedin_2023_2024_big,Respiratory Therapist,en,81733,100.000000,en,exact,47
9,2264355,job_ad,kaggle_linkedin_2023_2024_big,Worship Leader,en,24422,90.000000,en,fuzzy,44


## 10. Long Format: Doc × Skill (+ Join Display Labels)

- Each row corresponds to (doc_id, kldb_5_code, skill_id)
- This counts:
  - `skill_count`: how often a skill appears in the skill lists (number of rows)
  - `doc_freq`: in how many different documents the skill appears
Additionally, join display labels (`display_label`, `display_source`, `display_lang`).

In [32]:
# Long-form: Only relevant documents
doc_ok = df_docs_run[(df_docs_run["has_kldb_match"]==True) & (df_docs_run["n_skills"]>0)].copy()

doc_skills_long = (
    doc_ok[[
        "doc_id","source_type","source_name","job_title_raw","language_guess","kldb_5_code","kldb_match_title","kldb_match_score","match_lang","match_method","skills_rule_ids"
    ]]
    .explode("skills_rule_ids")
    .rename(columns={"skills_rule_ids":"skill_id"})
    .dropna(subset=["skill_id"])
    .reset_index(drop=True)
)

print("doc_skills_long shape:", doc_skills_long.shape)
display(doc_skills_long.head(20))

doc_skills_long shape: (9062244, 11)


,doc_id,source_type,source_name,job_title_raw,language_guess,kldb_5_code,kldb_match_title,kldb_match_score,match_lang,match_method,skill_id
0,921716,job_ad,kaggle_linkedin_2023_2024_big,Marketing Coordinator,en,92113,Marketing Coordinator,100.0,en,exact,BA:K 030203-027
1,921716,job_ad,kaggle_linkedin_2023_2024_big,Marketing Coordinator,en,92113,Marketing Coordinator,100.0,en,exact,BA:K 030203-054
2,921716,job_ad,kaggle_linkedin_2023_2024_big,Marketing Coordinator,en,92113,Marketing Coordinator,100.0,en,exact,BA:K 030203-062
3,921716,job_ad,kaggle_linkedin_2023_2024_big,Marketing Coordinator,en,92113,Marketing Coordinator,100.0,en,exact,BA:K 030203-065
4,921716,job_ad,kaggle_linkedin_2023_2024_big,Marketing Coordinator,en,92113,Marketing Coordinator,100.0,en,exact,BA:K 030300-025
5,921716,job_ad,kaggle_linkedin_2023_2024_big,Marketing Coordinator,en,92113,Marketing Coordinator,100.0,en,exact,BA:K 0700-049
6,921716,job_ad,kaggle_linkedin_2023_2024_big,Marketing Coordinator,en,92113,Marketing Coordinator,100.0,en,exact,BA:K 070100-129
7,921716,job_ad,kaggle_linkedin_2023_2024_big,Marketing Coordinator,en,92113,Marketing Coordinator,100.0,en,exact,BA:K 070104-010
8,921716,job_ad,kaggle_linkedin_2023_2024_big,Marketing Coordinator,en,92113,Marketing Coordinator,100.0,en,exact,BA:K 070104-022
9,921716,job_ad,kaggle_linkedin_2023_2024_big,Marketing Coordinator,en,92113,Marketing Coordinator,100.0,en,exact,BA:K 070104-076


In [33]:
# Display-Join, df_skill_display with columns: skill_id, display_label, display_source, display_lang
doc_skills_long = doc_skills_long.merge(
    df_skill_display[["skill_id","display_label","display_source","display_lang"]],
    on="skill_id", how="left"
)

display(doc_skills_long.head(20))

,doc_id,source_type,source_name,job_title_raw,language_guess,kldb_5_code,kldb_match_title,kldb_match_score,match_lang,match_method,skill_id,display_label,display_source,display_lang
0,921716,job_ad,kaggle_linkedin_2023_2024_big,Marketing Coordinator,en,92113,Marketing Coordinator,100.0,en,exact,BA:K 030203-027,Vertriebsmarketing,BA,de
1,921716,job_ad,kaggle_linkedin_2023_2024_big,Marketing Coordinator,en,92113,Marketing Coordinator,100.0,en,exact,BA:K 030203-054,Newsletter-Marketing,BA,de
2,921716,job_ad,kaggle_linkedin_2023_2024_big,Marketing Coordinator,en,92113,Marketing Coordinator,100.0,en,exact,BA:K 030203-062,Green Marketing,BA,de
3,921716,job_ad,kaggle_linkedin_2023_2024_big,Marketing Coordinator,en,92113,Marketing Coordinator,100.0,en,exact,BA:K 030203-065,Digital-Marketing,BA,de
4,921716,job_ad,kaggle_linkedin_2023_2024_big,Marketing Coordinator,en,92113,Marketing Coordinator,100.0,en,exact,BA:K 030300-025,Management,BA,de
5,921716,job_ad,kaggle_linkedin_2023_2024_big,Marketing Coordinator,en,92113,Marketing Coordinator,100.0,en,exact,BA:K 0700-049,Microsoft Office,BA,de
6,921716,job_ad,kaggle_linkedin_2023_2024_big,Marketing Coordinator,en,92113,Marketing Coordinator,100.0,en,exact,BA:K 070100-129,CAD-Anwendung AutoCAD Architecture,BA,de
7,921716,job_ad,kaggle_linkedin_2023_2024_big,Marketing Coordinator,en,92113,Marketing Coordinator,100.0,en,exact,BA:K 070104-010,DTP-Anwendung Adobe InDesign,BA,de
8,921716,job_ad,kaggle_linkedin_2023_2024_big,Marketing Coordinator,en,92113,Marketing Coordinator,100.0,en,exact,BA:K 070104-022,Grafikprogramm Adobe Illustrator,BA,de
9,921716,job_ad,kaggle_linkedin_2023_2024_big,Marketing Coordinator,en,92113,Marketing Coordinator,100.0,en,exact,BA:K 070104-076,Adobe Muse,BA,de


## 11. Aggregation: Which skills appear per KldB (Docs + Doc Frequency)

Aggregation of the long-form table at the KldB level: Per `(kldb_5_code, skill_id)`:
- `skill_count`: how often (number of rows) the skill occurs
- `doc_freq`: in how many different documents the skill occurs
- `example_label`/`example_source`: for better interpretation
- `doc_freq`
    - `skill_count` can be skewed by long lists of skills in individual documents
    - `doc_freq` is more robust because it counts across documents
- If a skill appears in only 1 document, it is more likely to be noise (but could also be a genuine new skill).

In [34]:
# Aggregation
agg_counts = (
    doc_skills_long
    .groupby(["kldb_5_code","skill_id"], as_index=False)
    .agg(
        skill_count=("doc_id","count"), # wie viele Doc×Skill-Zeilen
        doc_freq=("doc_id","nunique"), # wie viele unterschiedliche Docs
        example_label=("display_label","first"),
        example_source=("display_source","first"),
        example_lang=("display_lang","first"),
    )
)

print("agg_counts shape:", agg_counts.shape)
display(agg_counts.sort_values(["doc_freq","skill_count"], ascending=False).head(20))

agg_counts shape: (999601, 7)


,kldb_5_code,skill_id,skill_count,doc_freq,example_label,example_source,example_lang
190194,27103,LINKEDIN:html,8706,8706,HTML,LINKEDIN,en
186109,27103,BA:K 0705-052,8276,8276,Programmiersprache JavaScript,BA,de
186107,27103,BA:K 0705-050,7689,7689,Programmiersprache Java,BA,de
186167,27103,BA:K 0705-138,7584,7584,JavaScript-Framework jQuery,BA,de
186100,27103,BA:K 0705-039,7127,7127,Stylesheet-Sprache CSS,BA,de
194279,27103,LINKEDIN:web,7098,7098,Web,LINKEDIN,en
194493,27103,LINKEDIN:xml,6823,6823,XML,LINKEDIN,en
186104,27103,BA:K 0705-045,6539,6539,"HTML, XML, XHTML, XAML, XSLT",BA,de
187024,27103,ESCO:http://data.europa.eu/esco/skill/b4dc6e4f-dc7d-445f-8ce2-d7b9d225e282,6485,6485,AJAX,ESCO,en
185864,27103,BA:K 0702-076,6467,6467,Versionsverwaltungsprogramm Git,BA,de


In [35]:
# Filter, marked as new to avoid too much noise (if skills appear only once)
MIN_DOC_FREQ = 2  # 1 im Testlauf, 2 im Full Run
agg_counts_f = agg_counts[agg_counts["doc_freq"] >= MIN_DOC_FREQ].copy()

print("Filtered agg_counts:", agg_counts_f.shape, " (MIN_DOC_FREQ =", MIN_DOC_FREQ, ")")
display(agg_counts_f.sort_values(["doc_freq","skill_count"], ascending=False).head(20))

Filtered agg_counts: (566843, 7)  (MIN_DOC_FREQ = 2 )


,kldb_5_code,skill_id,skill_count,doc_freq,example_label,example_source,example_lang
190194,27103,LINKEDIN:html,8706,8706,HTML,LINKEDIN,en
186109,27103,BA:K 0705-052,8276,8276,Programmiersprache JavaScript,BA,de
186107,27103,BA:K 0705-050,7689,7689,Programmiersprache Java,BA,de
186167,27103,BA:K 0705-138,7584,7584,JavaScript-Framework jQuery,BA,de
186100,27103,BA:K 0705-039,7127,7127,Stylesheet-Sprache CSS,BA,de
194279,27103,LINKEDIN:web,7098,7098,Web,LINKEDIN,en
194493,27103,LINKEDIN:xml,6823,6823,XML,LINKEDIN,en
186104,27103,BA:K 0705-045,6539,6539,"HTML, XML, XHTML, XAML, XSLT",BA,de
187024,27103,ESCO:http://data.europa.eu/esco/skill/b4dc6e4f-dc7d-445f-8ce2-d7b9d225e282,6485,6485,AJAX,ESCO,en
185864,27103,BA:K 0702-076,6467,6467,Versionsverwaltungsprogramm Git,BA,de


## 12. Comparison with Baseline Target: Identifying Novel Skills

Load the base target profile and build a set of target skills for each KldB: `soll_sets[kldb_5_code] = {skill_id, ...}`
Mark the aggregated skill as:
- new (`is_novel_vs_soll = True`) if it is not in the target set,
- otherwise, not new.
Actual step for profile expansion: add external skills as candidates in addition to the baseline target set.

In [36]:
# Convert target skill IDs to the skill_id format
import numpy as np
import pandas as pd

def pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def to_skill_id(x):
    x = str(x)
    # Prefixe
    if x.startswith(("ESCO:", "BA:", "LINKEDIN:")):
        return x
    # target Skill URIs
    if x.startswith("http"):
        return "ESCO:" + x
    return x

# load target Long 
df_soll = pd.read_parquet(KDB_SOLL_LONG_PATH)  # Path
skill_col = pick_col(df_soll, ["skill_uri","skill_id","skill"])
assert skill_col is not None, "Kein Skill-Feld im SOLL gefunden!"

df_soll_tmp = df_soll[["kldb_5_code", skill_col]].dropna().copy()
df_soll_tmp["kldb_5_code"] = df_soll_tmp["kldb_5_code"].astype(str).str.extract(r"(\d{5})")[0]
df_soll_tmp["skill_id_soll"] = df_soll_tmp[skill_col].apply(to_skill_id)

soll_sets = df_soll_tmp.groupby("kldb_5_code")["skill_id_soll"].apply(lambda s: set(s)).to_dict()
print("SOLL sets built for codes:", len(soll_sets))

SOLL sets built for codes: 1287


In [37]:
# Mark as new
ext = agg_counts_f.copy()
ext["is_novel_vs_soll"] = ext.apply(
    lambda r: r["skill_id"] not in soll_sets.get(r["kldb_5_code"], set()),
    axis=1
)

ext_new = ext[ext["is_novel_vs_soll"]==True].copy()
# Keep only new skills (final safety filter)
ext_new = ext_new[~ext_new.apply(lambda r: r["skill_id"] in soll_sets.get(r["kldb_5_code"], set()), axis=1)].copy()

# Temporarily exclude LinkedIn skills; this was done first, and the documents were saved, but not as the final version
ext_new = ext_new[ext_new["example_source"] != "LINKEDIN"].copy()

print("Novel skills rows:", ext_new.shape)
display(ext_new.sort_values(["doc_freq","skill_count"], ascending=False).head(25))

Novel skills rows: (149998, 8)


,kldb_5_code,skill_id,skill_count,doc_freq,example_label,example_source,example_lang,is_novel_vs_soll
186109,27103,BA:K 0705-052,8276,8276,Programmiersprache JavaScript,BA,de,True
186107,27103,BA:K 0705-050,7689,7689,Programmiersprache Java,BA,de,True
186167,27103,BA:K 0705-138,7584,7584,JavaScript-Framework jQuery,BA,de,True
186100,27103,BA:K 0705-039,7127,7127,Stylesheet-Sprache CSS,BA,de,True
186104,27103,BA:K 0705-045,6539,6539,"HTML, XML, XHTML, XAML, XSLT",BA,de,True
187024,27103,ESCO:http://data.europa.eu/esco/skill/b4dc6e4f-dc7d-445f-8ce2-d7b9d225e282,6485,6485,AJAX,ESCO,en,True
185864,27103,BA:K 0702-076,6467,6467,Versionsverwaltungsprogramm Git,BA,de,True
187127,27103,ESCO:http://data.europa.eu/esco/skill/e5d1f825-60ed-4bdd-872a-e748c387f777,6248,6248,CSS,ESCO,en,True
698064,72144,BA:K 0711-021,5834,5834,Informatik,BA,de,True
186219,27103,BA:K 0705-206,5760,5760,Spark Framework,BA,de,True


## 13. Result: Generate Expanded Target Profiles (Long & Agg)

Some unexpected skills—such as very generic LinkedIn skills—sometimes appear in the Novel Skills table. Typical causes:
- External texts contain standard phrases (soft skills/employer branding)
- LinkedIn skills are often broad and lack context
- Job title matching is heuristic (even with a score ≥ cutoff, an incorrect match may occur)

Save at the end:
- Doc-level result
- Aggregation per KldB
- New skills table
- Extended Target (Long) and Agg

Attempts Made/Possible Attempts to Improve Quality:
1. Minimum threshold: `doc_freq >= 2` or `>= 3`, especially for a full run; here, 2 is used for broader/more results
2. Top-N per KldB: only the top 20–50 skills
3. Stoplist for very generic skills (e.g., “written communication,” “teams,” …)
4. Source weighting: e.g., give job ads more weight than profiles
5. Slightly increase the cutoff for cleaner matches, resulting in fewer matches
6. Further adjustments/extensions to external datasets or the skill vocabulary

In [38]:
# expand target Long 
df_soll_long_base = df_soll.copy()
df_soll_long_base["kldb_5_code"] = df_soll_long_base["kldb_5_code"].astype(str).str.extract(r"(\d{5})")[0]
df_soll_long_base["skill_id"] = df_soll_long_base[skill_col].apply(to_skill_id)

# Baseline
df_soll_long_base["is_extension"] = False
df_soll_long_base["extension_method"] = None
df_soll_long_base["ext_doc_freq"] = np.nan
df_soll_long_base["ext_skill_count"] = np.nan
df_soll_long_base["ext_example_label"] = None
df_soll_long_base["ext_example_source"] = None

# Extension rows
ext_append = ext_new[["kldb_5_code","skill_id","doc_freq","skill_count","example_label","example_source"]].copy()
ext_append["is_extension"] = True
ext_append["extension_method"] = "rule_based_1_1"
ext_append["ext_doc_freq"] = ext_append["doc_freq"]
ext_append["ext_skill_count"] = ext_append["skill_count"]
ext_append["ext_example_label"] = ext_append["example_label"]
ext_append["ext_example_source"] = ext_append["example_source"]

need_cols = ["kldb_5_code","skill_id","is_extension","extension_method",
             "ext_doc_freq","ext_skill_count","ext_example_label","ext_example_source"]

for c in need_cols:
    if c not in df_soll_long_base.columns:
        df_soll_long_base[c] = np.nan

# Expansion
base_pairs = set(zip(df_soll_long_base["kldb_5_code"], df_soll_long_base["skill_id"]))

ext_append = ext_append[~ext_append.apply(lambda r: (r["kldb_5_code"], r["skill_id"]) in base_pairs, axis=1)].copy()
df_soll_long_extended = pd.concat([df_soll_long_base, ext_append[need_cols]], ignore_index=True)

print("Extended SOLL long shape:", df_soll_long_extended.shape)

Extended SOLL long shape: (959410, 27)


In [39]:
# expand target Agg 
KDB_SOLL_AGG_PATH = DATA_PROCESSED / "kldb_skills_soll_agg.parquet"
df_soll_agg = pd.read_parquet(KDB_SOLL_AGG_PATH)
df_soll_agg["skills"] = df_soll_agg["skills"].apply(lambda x: x.tolist() if hasattr(x, "tolist") else x)
df_soll_agg["kldb_5_code"] = df_soll_agg["kldb_5_code"].astype(str).str.extract(r"(\d{5})")[0]

def extract_ids_from_skills(skills_obj):
    if skills_obj is None:
        return []
    if hasattr(skills_obj, "tolist"):   # numpy array -> list
        skills_obj = skills_obj.tolist()

    if not isinstance(skills_obj, list):
        return []

    out = []
    for it in skills_obj:
        if isinstance(it, dict):
            v = it.get("skill_uri") or it.get("skill_id") or it.get("skill")
            if v: out.append(to_skill_id(v))
        else:
            out.append(to_skill_id(it))
    return sorted(set(out))

ext_list = (
    ext_new.groupby("kldb_5_code")["skill_id"]
    .apply(lambda s: sorted(set(s)))
    .reset_index(name="skills_extension")
)

df_soll_agg_ext = df_soll_agg.merge(ext_list, on="kldb_5_code", how="left")
df_soll_agg_ext["skills_extension"] = df_soll_agg_ext["skills_extension"].apply(lambda x: x if isinstance(x, list) else [])
df_soll_agg_ext["skills_base_ids"] = df_soll_agg_ext["skills"].apply(extract_ids_from_skills)

df_soll_agg_ext["skills_all_ids"] = df_soll_agg_ext.apply(
    lambda r: sorted(set(r["skills_base_ids"]) | set(r["skills_extension"])),
    axis=1
)

print("Extended SOLL agg shape:", df_soll_agg_ext.shape)
display(df_soll_agg_ext[["kldb_5_code","skills_extension"]].head(10))

Extended SOLL agg shape: (1300, 8)


,kldb_5_code,skills_extension
0,01104,[]
1,01203,"[BA:K 010205-001, BA:K 030300-025, BA:K 0700-010, BA:K 0700-030, BA:K 0702-017, BA:K 0805-034, BA:K 090001-009, BA:K..."
2,01302,[]
3,01402,"[BA:K 010205-001, BA:K 010401-028, BA:K 010601-002, BA:K 010903-025, BA:K 030000-023, BA:K 030000-052, BA:K 030203-0..."
4,11101,"[BA:K 0004, BA:K 0004-010, BA:K 010203-004, BA:K 010203-006, BA:K 010203-008, BA:K 010203-019, BA:K 010205-001, BA:K..."
5,11102,"[BA:K 010202-038, BA:K 010203-006, BA:K 010203-019, BA:K 010205-001, BA:K 0209-041, BA:K 030000-050, BA:K 030000-052..."
6,11103,"[BA:K 010203-004, BA:K 010203-006, BA:K 010205-001, BA:K 010400-049, BA:K 010601-002, BA:K 010800-030, BA:K 010800-0..."
7,11104,"[BA:K 010205-001, BA:K 010400-049, BA:K 010404-003, BA:K 0106, BA:K 010800-030, BA:K 010800-044, BA:K 010900-000, BA..."
8,11113,"[BA:K 010203-004, BA:K 010205-001, BA:K 010903-025, BA:K 030206-004, BA:K 030300-025, BA:K 030300-037, BA:K 030300-0..."
9,11114,[]


## 14. Saving the Results

Doc-Level, Aggregation, Extension, Extended SOLL: separately for with and without LinkedIn as a skill source (with LinkedIn in the “processed_external/10b Results with LinkedIn” subfolder)

In [40]:
# Output paths
DOC_SKILLS_FULL_PATH    = DATA_PROCESSED_EXTERNAL / "doc_skills_rulebased_full.parquet"
KDB_SKILLS_FULL_PATH    = DATA_PROCESSED_EXTERNAL / "kldb_skills_rulebased_full.parquet"
KDB_EXTENSION_FULL_PATH = DATA_PROCESSED_EXTERNAL / "kldb_skill_rule_extension_full.parquet"

SOLL_LONG_EXT_PATH = DATA_PROCESSED_EXTERNAL / "kldb_skills_soll_long_extended_rulebased.parquet"
SOLL_AGG_EXT_PATH  = DATA_PROCESSED_EXTERNAL / "kldb_skills_soll_agg_extended_rulebased.parquet"

# Doc-level
doc_cols_keep = ["doc_id","source_type","source_name","job_title_raw","language","language_guess","kldb_5_code","kldb_match_title","kldb_match_score","has_kldb_match","match_lang","match_method","n_skills"]
df_docs_out = df_docs_run[[c for c in doc_cols_keep if c in df_docs_run.columns]].copy()

# Save
df_docs_out.to_parquet(DOC_SKILLS_FULL_PATH, index=False)
agg_counts_f.to_parquet(KDB_SKILLS_FULL_PATH, index=False)
ext_new.to_parquet(KDB_EXTENSION_FULL_PATH, index=False)
df_soll_long_extended.to_parquet(SOLL_LONG_EXT_PATH, index=False)

print("Saved:")
print(" -", DOC_SKILLS_FULL_PATH)
print(" -", KDB_SKILLS_FULL_PATH)
print(" -", KDB_EXTENSION_FULL_PATH)
print(" -", SOLL_LONG_EXT_PATH)

# agg save
if "df_soll_agg_ext" in globals():
    df_soll_agg_ext.to_parquet(SOLL_AGG_EXT_PATH, index=False)
    print(" -", SOLL_AGG_EXT_PATH)

Saved:
 - c:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_external\doc_skills_rulebased_full.parquet
 - c:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_external\kldb_skills_rulebased_full.parquet
 - c:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_external\kldb_skill_rule_extension_full.parquet
 - c:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_external\kldb_skills_soll_long_extended_rulebased.parquet
 - c:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_external\kldb_skills_soll_agg_extended_rulebased.parquet


Saved files:

1) Doc-Level Output (`doc_skills_rulebased_full.parquet`), contains the following for each document:
- `doc_id`, `source_type`, `source_name`, `job_title_raw`
- Matching results: `kldb_5_code`, `kldb_match_score`, `has_kldb_match`, `match_lang`, `match_method`
- Scope of extraction: `n_skills`

2) Aggregation per KldB (`kldb_skills_rulebased_full.parquet`), long-form/aggregation view for each `(kldb_5_code, skill_id)`:
- `doc_freq` (robust): Documents containing this skill
- `skill_count` (intensive): Occurrences in long-form
- Example information: `example_label`, `example_source`, `example_lang`

3) Novel/Extension (`kldb_skill_rule_extension_full.parquet`), subset of the aggregation:
- only skills not included in the baseline target (`is_novel_vs_soll=True`)

4) Extended target Long (`kldb_skills_soll_long_extended_rulebased.parquet`), baseline target in long format + extension rows:
- Marker: `is_extension`, `extension_method`
- Meta: `ext_doc_freq`, `ext_skill_count`, `ext_example_label`, `ext_example_source`

5) Extended target Agg (`kldb_skills_soll_agg_extended_rulebased.parquet`), aggregated view:
- `skills_base_ids` (from Baseline Target)
- `skills_extension` (new skills)
- `skills_all_ids` (Union)

## 15. Sample Outputs + Checks

In [41]:
# Check: Top KldB Codes Based on new skills
top_codes = ext_new.groupby("kldb_5_code")["skill_id"].nunique().sort_values(ascending=False).head(10)
print("Top codes by novel skills:")
display(top_codes)

if len(ext_new):
    top_code = top_codes.index[1] # KLDB code is:
    print("Example one of top_code:", top_code)
    display(ext_new[ext_new["kldb_5_code"]==top_code].sort_values(["doc_freq","skill_count"], ascending=False).head(25))

Top codes by novel skills:


kldb_5_code
72144    1694
27103    1529
63301    1473
92113    1439
28213    1423
43414    1305
11184    1187
84223    1165
43343    1141
11183    1123
Name: skill_id, dtype: int64

Example one of top_code: 27103


,kldb_5_code,skill_id,skill_count,doc_freq,example_label,example_source,example_lang,is_novel_vs_soll
186109,27103,BA:K 0705-052,8276,8276,Programmiersprache JavaScript,BA,de,True
186107,27103,BA:K 0705-050,7689,7689,Programmiersprache Java,BA,de,True
186167,27103,BA:K 0705-138,7584,7584,JavaScript-Framework jQuery,BA,de,True
186100,27103,BA:K 0705-039,7127,7127,Stylesheet-Sprache CSS,BA,de,True
186104,27103,BA:K 0705-045,6539,6539,"HTML, XML, XHTML, XAML, XSLT",BA,de,True
187024,27103,ESCO:http://data.europa.eu/esco/skill/b4dc6e4f-dc7d-445f-8ce2-d7b9d225e282,6485,6485,AJAX,ESCO,en,True
185864,27103,BA:K 0702-076,6467,6467,Versionsverwaltungsprogramm Git,BA,de,True
187127,27103,ESCO:http://data.europa.eu/esco/skill/e5d1f825-60ed-4bdd-872a-e748c387f777,6248,6248,CSS,ESCO,en,True
186219,27103,BA:K 0705-206,5760,5760,Spark Framework,BA,de,True
186110,27103,BA:K 0705-054,5506,5506,Programmiersprache JSP (Java Server Pages),BA,de,True


The top score with new skills is around 1,700 here. While that’s also very high, it further confirms the impact of LinkedIn skills, which actually brought the total to 9,000. That makes it no longer realistic. BA & ESCO can, of course, also contain simpler, shorter, and more generic skills that are then found very often in external datasets; these are at least standardized, but the situation is even more extreme on LinkedIn. With a cutoff of 0.90, similar to 0.85, the result is approximately 1,700. This makes it more plausible:
- Main variant: LinkedIn skills excluded because they are too generic/noisy.
- Reference run: LinkedIn variant saved separately (subfolder 10b: results with LinkedIn)

In [42]:
# All KldB-Codes
all_codes = df_candidates["kldb_5_code"].dropna().unique()
# KldB-Codes that have Novel-Skills 
codes_with_novel = ext_new["kldb_5_code"].dropna().unique()
# Codes without Novel-Skills
codes_no_novel = sorted(set(all_codes) - set(codes_with_novel))

print("Anzahl KldB-Codes insgesamt:", len(all_codes))
print("Anzahl KldB-Codes mit neuen Skills:", len(codes_with_novel))
print("Anzahl KldB-Codes ohne neue Skills:", len(codes_no_novel))

# Sample output (first 10 codes without Novel Skills)
example_codes = codes_no_novel[:10]
print("Beispiele:", example_codes)

# Map titles
cols = ["kldb_5_code"]
if "job_title" in df_candidates.columns:
    cols.append("job_title")

example_df = (
    df_candidates[df_candidates["kldb_5_code"].isin(example_codes)][cols]
    .drop_duplicates(subset=["kldb_5_code"])
    .sort_values("kldb_5_code")
)
display(example_df)

# Number of unique Novel Skills per code
novel_per_code = (
    ext_new.groupby("kldb_5_code")["skill_id"]
    .nunique()
    .sort_values(ascending=False)
)

print("\nNeue Skills pro erweiterter KldB-Code (unique skill_id)")
print("Anzahl erweiterter Codes:", novel_per_code.shape[0])
display(novel_per_code.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]))

Anzahl KldB-Codes insgesamt: 1300
Anzahl KldB-Codes mit neuen Skills: 893
Anzahl KldB-Codes ohne neue Skills: 407
Beispiele: ['01104', '01302', '11114', '11123', '11133', '11182', '11211', '11214', '11222', '11223']


,kldb_5_code,job_title
12433,01104,Offizier - Geoinformationsdienst
5176,01302,Fachunteroffizier - Allgemeiner Fachdienst
3253,11114,Dipl.-Ing. (Ing.H) - Mechanisierung der Pflanzenproduktion
10123,11123,Landwirtschaftliche/r Gutachter/in
10124,11133,Landwirtschaftliche/r Labortechniker/in
122,11182,Agrochemiker/in
6313,11211,Ftterer/Ftterin - Tierwirtschaft
2953,11214,Dipl.-Agraring./Dipl.Landwirt/in (Uni) - Tierproduktion
4539,11222,Facharbeiter/in - Geflgelproduktion
4770,11223,Fachberater/in - Geflgelzucht



Neue Skills pro erweiterter KldB-Code (unique skill_id)
Anzahl erweiterter Codes: 893


count     893.000000
mean      167.970885
std       236.075051
min         1.000000
10%        10.000000
25%        26.000000
50%        83.000000
75%       203.000000
90%       428.800000
max      1694.000000
Name: skill_id, dtype: float64

Approximately 900 KldB codes were expanded, while 400 did not receive any new skills. With Min_Doc_Freq = 1, more codes would be expanded, but the skills would then have only one external source as a basis for matching. Since the numbers are very similar even with a score cutoff of 0.80, we set the cutoff here to 0.85 - preferring a slightly higher threshold to ensure more relevant matches. With LinkedIn, there are about 600 new skills on average; here, “only” 170. With a cutoff of 0.90, only about half of the 1,300 KldB codes are expanded, with an average of 124 skills. That’s fewer, which is why we use 0.85.

Lower KldB code coverage may also be related to the more specific origins or domains of the external datasets.

Sample output, complete KldB code 28213 (Fashion Design Occupations), target Basic Profile + Extension:

In [43]:
code = "28213"
tmp = df_soll_agg_ext[df_soll_agg_ext["kldb_5_code"].astype(str).str.zfill(5) == str(code).zfill(5)].head(1)

display(tmp[["kldb_5_code", "kldb_title_de", "kldb_title_en"]].T if len(tmp) else tmp)

if len(tmp):
    skills_obj = tmp.iloc[0].get("skills")
    print("type(skills) =", type(skills_obj))
    print("len(skills)  =", len(skills_obj) if isinstance(skills_obj, list) else "n/a")
    if isinstance(skills_obj, list) and skills_obj:
        print("first item type =", type(skills_obj[0]))
        print("first item keys =", list(skills_obj[0].keys()) if isinstance(skills_obj[0], dict) else "n/a")

,341
kldb_5_code,28213
kldb_title_de,Berufe im Modedesign - komplexe Spezialistentätigkeiten
kldb_title_en,Occupations in fashion design-complex tasks


type(skills) = <class 'list'>
len(skills)  = 579
first item type = <class 'dict'>
first item keys = ['relation_type', 'reuse_level', 'skill_title_en', 'skill_type', 'skill_uri']


In [44]:
# Path
KDB_SOLL_LONG_PATH = DATA_PROCESSED / "kldb_skills_soll_long.parquet"

df_soll_long = pd.read_parquet(KDB_SOLL_LONG_PATH).copy()
df_soll_long["kldb_5_code"] = df_soll_long["kldb_5_code"].astype(str).str.extract(r"(\d{5})")[0]
df_soll_long["skill_id"] = df_soll_long["skill_uri"].apply(to_skill_id)

# Build Extension-Long from ext_new
ext_long = ext_new.copy()
ext_long["kldb_5_code"] = ext_long["kldb_5_code"].astype(str).str.zfill(5)
ext_long["skill_id"] = ext_long["skill_id"].apply(to_skill_id)

# Remove duplicates: Extension skills that are already included in the baseline target should be removed, just to be safe
base_set = (
    df_soll_long.groupby("kldb_5_code")["skill_id"].apply(lambda s: set(s.dropna())).to_dict()
)

ext_long["already_in_base"] = ext_long.apply(
    lambda r: r["skill_id"] in base_set.get(r["kldb_5_code"], set()),
    axis=1
)
ext_long_clean = ext_long[~ext_long["already_in_base"]].copy()

# Sample display of a code
def show_profile_long(kldb_code: str, n_base=20, n_ext=15):
    kldb_code = str(kldb_code).zfill(5)

    base_rows = (
        df_soll_long[df_soll_long["kldb_5_code"] == kldb_code]
        .loc[:, ["kldb_5_code","kldb_title_de","kldb_title_en","skill_id","skill_title_en","relation_type","skill_type","reuse_level"]]
        .drop_duplicates(subset=["kldb_5_code","skill_id"]).head(n_base)
    )

    ext_rows = (
        ext_long_clean[ext_long_clean["kldb_5_code"] == kldb_code]
        .loc[:, ["kldb_5_code","skill_id","doc_freq","skill_count","example_label","example_source","example_lang"]]
        .drop_duplicates(subset=["kldb_5_code","skill_id"])
        .sort_values(["doc_freq","skill_count"], ascending=False)
        .head(n_ext)
    )

    print("KldB:", kldb_code)
    print("Base skills (unique):", df_soll_long[df_soll_long["kldb_5_code"] == kldb_code]["skill_id"].nunique())
    print("Extension skills (unique, deduped):", ext_long_clean[ext_long_clean["kldb_5_code"] == kldb_code]["skill_id"].nunique())
    print("-"*60)
    print("BASE (sample)")
    display(base_rows)
    print("EXTENSION (sample)")
    display(ext_rows)

# Example:
show_profile_long("28213", n_base=20, n_ext=15)

KldB: 28213
Base skills (unique): 356
Extension skills (unique, deduped): 1423
------------------------------------------------------------
BASE (sample)


,kldb_5_code,kldb_title_de,kldb_title_en,skill_id,skill_title_en,relation_type,skill_type,reuse_level
229989,28213,Berufe im Modedesign - komplexe Spezialistentätigkeiten,Occupations in fashion design-complex tasks,ESCO:http://data.europa.eu/esco/skill/0cf4c414-891e-4030-a3c3-898643bf20fc,footwear and leather goods marketing planning,essential,knowledge,sector-specific
229990,28213,Berufe im Modedesign - komplexe Spezialistentätigkeiten,Occupations in fashion design-complex tasks,ESCO:http://data.europa.eu/esco/skill/a4cfad38-109a-4a61-8d01-98fd8cf001ce,footwear materials,essential,knowledge,sector-specific
229991,28213,Berufe im Modedesign - komplexe Spezialistentätigkeiten,Occupations in fashion design-complex tasks,ESCO:http://data.europa.eu/esco/skill/c6846331-2e11-45d6-ab8d-306c956332fc,footwear manufacturing technology,essential,knowledge,sector-specific
229992,28213,Berufe im Modedesign - komplexe Spezialistentätigkeiten,Occupations in fashion design-complex tasks,ESCO:http://data.europa.eu/esco/skill/de278695-33e1-400a-8962-95f0ba5ce2e0,footwear creation process,essential,knowledge,sector-specific
229993,28213,Berufe im Modedesign - komplexe Spezialistentätigkeiten,Occupations in fashion design-complex tasks,ESCO:http://data.europa.eu/esco/skill/e54a3ed7-465f-432b-8940-2aeabeaa449e,footwear quality,essential,knowledge,sector-specific
229994,28213,Berufe im Modedesign - komplexe Spezialistentätigkeiten,Occupations in fashion design-complex tasks,ESCO:http://data.europa.eu/esco/skill/e6b31fc5-97c4-45b3-bb5b-5c93e5711d5d,ergonomics in footwear and leather goods design,essential,knowledge,sector-specific
229995,28213,Berufe im Modedesign - komplexe Spezialistentätigkeiten,Occupations in fashion design-complex tasks,ESCO:http://data.europa.eu/esco/skill/ec7fd241-e68f-4593-a42a-a12a9bd80cbc,footwear components,essential,knowledge,sector-specific
229996,28213,Berufe im Modedesign - komplexe Spezialistentätigkeiten,Occupations in fashion design-complex tasks,ESCO:http://data.europa.eu/esco/skill/f3880260-ef21-401c-a64f-a96f21dc968a,last types,essential,knowledge,sector-specific
229997,28213,Berufe im Modedesign - komplexe Spezialistentätigkeiten,Occupations in fashion design-complex tasks,ESCO:http://data.europa.eu/esco/skill/10311d30-7788-4866-be1c-8235832b7cee,create patterns for footwear,essential,skill/competence,sector-specific
229998,28213,Berufe im Modedesign - komplexe Spezialistentätigkeiten,Occupations in fashion design-complex tasks,ESCO:http://data.europa.eu/esco/skill/11a76a95-66c1-43e0-91f0-be1cc5d05964,innovate in footwear and leather goods industry,essential,skill/competence,cross-sector


EXTENSION (sample)


,kldb_5_code,skill_id,doc_freq,skill_count,example_label,example_source,example_lang
245854,28213,BA:K 070104-003,3908,3908,"Autorensysteme (Macromedia Director, Toolbook u.a.)",BA,de
246660,28213,ESCO:http://data.europa.eu/esco/skill/15d76317-c71a-4fa2-aadc-2ecc34e627b7,1942,1942,communication,ESCO,en
245666,28213,BA:K 030206-004,1927,1927,"Berichtswesen, Information",BA,de
246281,28213,BA:K 0711-021,1805,1805,Informatik,BA,de
245674,28213,BA:K 030300-025,1681,1681,Management,BA,de
245761,28213,BA:K 0700-010,1632,1632,First-Level-Support,BA,de
245681,28213,BA:K 030300-043,1617,1617,On-Site-Management,BA,de
246466,28213,BA:K 100008-014,1491,1491,Performance (Kunst),BA,de
247259,28213,ESCO:http://data.europa.eu/esco/skill/e49f4158-9d4c-425d-bf32-dfe89b19840a,1440,1440,plan,ESCO,en
245521,28213,BA:K 010205-001,1336,1336,Kundendienst,BA,de


Based on the various sample outputs, it can be noted that most of the extended skills are plausible. However, there are some short or very generic terms that were frequently added as extensions to many KldB codes (primarily from the LinkedIn skill dataset). For this reason, it was decided to exclude LinkedIn from the skill vocabulary for the main variant. Results including LinkedIn were nevertheless saved separately.

## 16. Preparation for the Gold Evaluation Set

For later evaluation (Notebook 08), the predictions from this method are exported into a standardized format. Standard export (per method):
- `doc_id`
- `skill_id` (normalized: ESCO/BA/LINKEDIN/...)
- `method` (`rule_based_1.1`)
- Metadata for traceability (e.g., `source_type`, `source_name`, `kldb_5_code`, `match_score`)

In [45]:
METHOD_NAME = "rule_based_1.1"
PRED_OUT_PATH = DATA_PROCESSED_EXTERNAL / "pred_skills_10b_rulebased.parquet"

# doc_skills_long
pred = doc_skills_long[["doc_id", "skill_id"]].dropna().drop_duplicates().copy()

# method column
pred["method"] = METHOD_NAME

# add Meta
meta_cols = [c for c in ["source_type","source_name","job_title_raw","language_guess","kldb_5_code","kldb_match_score","match_lang","match_method"] if c in doc_skills_long.columns]
if meta_cols:
    pred = pred.merge(
        doc_skills_long[["doc_id"] + meta_cols].drop_duplicates(subset=["doc_id"]),
        on="doc_id", how="left"
    )

pred.to_parquet(PRED_OUT_PATH, index=False)
print("Saved:", PRED_OUT_PATH, "| rows:", len(pred), "| unique docs:", pred["doc_id"].nunique())

Saved: c:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\processed_external\pred_skills_10b_rulebased.parquet | rows: 9062244 | unique docs: 187774


(saved separately for cases with and without LinkedIn as a skill source)

This will allow Notebook 08 to later:
- Load Gold Excel
- Load pred_skills_*.parquet for each method
- Compare precision, recall, F1, etc. (per document and overall)

# Conclusion Notebook 10_b

In this notebook, the first knowledge-based profile extension method (Rule-based Method 1.1) was successfully implemented. Building on the previously generated KldB-target profiles, external text sources were used to extract additional, potentially relevant skills and assign them to the existing base profiles. Rule-based and lexicon-based methods represent a transparent but content-limited baseline that is particularly well-suited as a starting point for advanced or hybrid methods (Cenikj et al. 2021). 

A key challenge lay in the scalable processing of large amounts of text under limited memory resources. Through a combination of:
- selective processing of only matched documents,
- record-group-based Arrow iteration,
- early truncation of long texts, and
- consistent skill ID normalization, a stable and fully operational methodology was implemented.

The results show that new skills for a subset of the KldB codes could already be identified during the test run. This confirms that:
- many newly extracted skills are plausible and job-relevant,
- at the same time, however, generic or context-poor skills may also occur, particularly with broadly defined job titles or marketing-heavy texts,
- heuristic job title matching is not error-free despite a score threshold, but can be mitigated through downstream aggregation and filtering mechanisms.

The explicit separation of basic skills (ESCO-target) and extended skills, as well as the union of both sets, results in a consistent, extended target profile for each KldB code. This forms a stable foundation. Overall, the notebook demonstrates that a rule-based profile extension based on external text sources is technically feasible, compatible in terms of content, and methodologically well-founded, even if complete precision in content can only be achieved through combination with additional methods.

Working well:
- The pipeline runs stably in both test mode and full run.
- Scores and accepted matches can be controlled in a transparent manner (cutoff).
- Output files are clearly defined and reusable.
- Results are sufficiently plausible.

Limitations (deliberately accepted):
- External skills contain noise and generic terms; therefore, results are stored both with and without LinkedIn in the skill source. The main variant is run without LinkedIn. Reference run: LinkedIn variant saved separately (subfolder 10b: Results with LinkedIn)
- Matching is heuristic (not perfect, but practical).
- Quality can potentially be improved via filters/thresholds without changing the basic principle.

**Distinction from the alternative notebooks and rationale for the main variant:** The alternative notebooks for 10b show that while higher cutoff values lead to very precise matches, they significantly limit the expansion of the KLDB codes, whereas lower values have the opposite effect. The main variant therefore takes a more balanced approach, enabling a reasonable balance between match quality and coverage. For this reason, the main methodology is retained, while the stricter and more moderate alternatives serve as supplementary comparison and validation approaches.